# Notebook 2 — Oracle Retrain on Retain Set (Paper Protocol)

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv:2604.08271v1)

**Purpose:** Train a fresh model from scratch on the **retain set only**, giving the
gold-standard upper bound for unlearning.  Results used in Table 1 of the paper.

**Protocol — matches paper exactly (Table 1 / Appendix A.3):**
- Forget set = **one entire class** (all ~5 000 training images of that class).
- Retain set = all training images from the remaining 9 classes.
- Loop over all 10 CIFAR-10 classes as the forget class; results are **averaged** (mean ± std).
- **Output accuracy** evaluated on the **held-out test set** (paper Appendix A.2).
- **Probe & NCC** features from the full 50 000-sample training set D = D_r ∪ D_f;
  evaluated on test-set forget / retain subsets (paper §3.2 / eq. 3).
- Oracle retrain: 200 epochs (Table 4 line 2314), same SGD + cosine schedule as θ_o.

**Prerequisites:** Run Notebook 1 first.  Set `CKPT_DATASET_DIR` to the dataset mount path.

**Outputs:** `oracle_class{c}_seed{seed}.pt` per run; `results_oracle_paper.csv` summary.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
# ══ SET THIS to the Kaggle dataset mount path containing theta_o checkpoints from Notebook 1 ══
CKPT_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-nb1-cifar100'

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break
assert config_path, f'Cannot find cmf_benchmark_config.json under {CKPT_DATASET_DIR}'
with open(config_path) as f: NB1_CFG = json.load(f)

DATASET     = NB1_CFG['dataset']       # 'cifar10'
ARCH        = NB1_CFG['arch']          # 'resnet18'
NUM_CLASSES = NB1_CFG['num_classes']   # 10
TEST_MODE   = NB1_CFG.get('test_mode', False)

# Paper protocol: sweep all 10 classes as forget class; one seed.
FORGET_CLASSES = NB1_CFG.get('forget_classes', ([0, 1, 2, 3, 5] if DATASET.lower() == 'cifar100' else list(range(NUM_CLASSES))))
SEEDS          = [0]

# Oracle retrain hyperparameters — same SGD + cosine schedule as theta_o (Table 4).
PT = NB1_CFG['pretrain']
LR_INIT       = PT['lr_init']       # 1e-2
WEIGHT_DECAY  = PT['weight_decay']  # 5e-4
MOMENTUM      = PT['momentum']      # 0.9
WARMUP_EPOCHS = PT['warmup_epochs']
MIN_LR        = PT['min_lr']
BATCH_SIZE    = PT['batch_size']    # 256
# Table 4: Retain-only Retrain CIFAR-10 = 200 epochs
EPOCHS   = 5 if TEST_MODE else 200
PATIENCE = PT['patience']

CKPT_ROOT = '/kaggle/working/checkpoints/oracle_paper'
os.makedirs(CKPT_ROOT, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DATASET={DATASET}  ARCH={ARCH}  NUM_CLASSES={NUM_CLASSES}  device={device}')
print(f'Forget classes: {FORGET_CLASSES}  EPOCHS={EPOCHS}')

In [ ]:
import torchvision, torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Training set — augmented (for training the oracle)
DatasetClass = torchvision.datasets.CIFAR100 if DATASET.lower() == 'cifar100' else torchvision.datasets.CIFAR10
full_train      = DatasetClass('/kaggle/working/data', train=True,
                                               download=True,  transform=transform_train)
# Training set — eval transform (no aug) — for probe/NCC feature extraction (D_r ∪ D_f)
full_train_eval = DatasetClass('/kaggle/working/data', train=True,
                                               download=False, transform=transform_test)
# Test set — for ALL final metric evaluation (paper Appendix A.2)
test_set = DatasetClass('/kaggle/working/data', train=False,
                                        download=True, transform=transform_test)

# Pre-index test set by class label for fast per-class loader construction.
test_targets = torch.tensor(test_set.targets)   # shape [10000]
TEST_CLASS_IDX = {
    c: (test_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}

# Pre-index training set by class label.
train_targets = torch.tensor(full_train.targets)  # shape [50000]
TRAIN_CLASS_IDX = {
    c: (train_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}

print(f'Train: {len(full_train)}  Test: {len(test_set)}')
print(f'Test samples per class: {len(TEST_CLASS_IDX[0])} (expected 1000 for CIFAR-10)')

In [ ]:
from models.resnet import ResNet18

def build_model():
    return ResNet18(num_classes=NUM_CLASSES, dataset=DATASET).to(device)

def make_scheduler(optimizer, warmup_epochs, total_epochs, min_lr):
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=min_lr)
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])


@torch.no_grad()
def eval_acc(model, loader):
    """Direct model accuracy on any loader."""
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)


@torch.no_grad()
def extract_features_resnet(model, loader):
    """Extract avgpool features via forward hook."""
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        buf = []
        hook = model.avgpool.register_forward_hook(
            lambda m, i, o: buf.append(o.flatten(1).detach().cpu())
        )
        model(x)
        hook.remove()
        feats.append(buf[0]); labs.append(y)
    return torch.cat(feats), torch.cat(labs)


def run_linear_probe(model, train_retain_ldr, train_forget_ldr,
                     test_retain_ldr, test_forget_ldr,
                     n_epochs=None, lr=1e-2):
    if n_epochs is None:
        n_epochs = 200 if DATASET.lower() == 'cifar100' else 50
    """
    Paper §3.2: probe trained on ALL training features D_r ∪ D_f,
    evaluated on test-set retain and test-set forget subsets.
    """
    Xtr, ytr = extract_features_resnet(model, train_retain_ldr)
    Xfg, yfg = extract_features_resnet(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])

    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=lr, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()

    with torch.no_grad():
        Xte_r, yte_r = extract_features_resnet(model, test_retain_ldr)
        Xte_f, yte_f = extract_features_resnet(model, test_forget_ldr)
        ret_acc = (head(Xte_r.to(device)).argmax(1).cpu() == yte_r).float().mean().item() * 100
        fgt_acc = (head(Xte_f.to(device)).argmax(1).cpu() == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc


def run_ncc(model, train_retain_ldr, train_forget_ldr,
            test_retain_ldr, test_forget_ldr):
    """
    Paper eq. 3: class means from ALL training samples D_r ∪ D_f.
    NCC accuracy evaluated on test-set retain and test-set forget subsets.
    """
    Xtr, ytr = extract_features_resnet(model, train_retain_ldr)
    Xfg, yfg = extract_features_resnet(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)  # [C, D]

    Xte_r, yte_r = extract_features_resnet(model, test_retain_ldr)
    Xte_f, yte_f = extract_features_resnet(model, test_forget_ldr)
    ret_pred = torch.cdist(Xte_r.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    fgt_pred = torch.cdist(Xte_f.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    ret_acc  = (ret_pred == yte_r).float().mean().item() * 100
    fgt_acc  = (fgt_pred == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc


def eval_three_metrics(model,
                       test_retain_ldr, test_forget_ldr,
                       train_retain_ldr, train_forget_ldr):
    """
    Three-metric evaluation matching paper Table 1 exactly.

    Output  — direct model accuracy on HELD-OUT TEST-SET retain / forget images.
    Probe   — linear head trained on ALL TRAINING features (D_r ∪ D_f),
              evaluated on test-set forget / retain.
    NCC     — class means from ALL TRAINING features, evaluated on test-set.
    """
    out_ret = eval_acc(model, test_retain_ldr)
    out_fgt = eval_acc(model, test_forget_ldr)

    lp_ret, lp_fgt = run_linear_probe(
        model,
        train_retain_ldr, train_forget_ldr,
        test_retain_ldr,  test_forget_ldr
    )

    ncc_ret, ncc_fgt = run_ncc(
        model,
        train_retain_ldr, train_forget_ldr,
        test_retain_ldr,  test_forget_ldr
    )

    return {
        'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
        'probe_retain_acc':  lp_ret,  'probe_forget_acc':  lp_fgt,
        'ncc_retain_acc':    ncc_ret, 'ncc_forget_acc':    ncc_fgt,
    }

print('Helpers ready.')

In [ ]:
all_results = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:

        # ── Build whole-class forget / retain splits from training set ──────────
        # Paper Appendix A.3: forget set = all samples of one class.
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]           # ~5 000 samples
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]                                                            # ~45 000 samples

        # Retain loader (augmented) — oracle trains on this ONLY
        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

        # Training-set loaders (eval transform) — for probe/NCC feature extraction
        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)

        # ── Test-set loaders — used for ALL final metric evaluation ──────────────
        # Paper Appendix A.2: "report performance on the test set".
        test_forget_idx = TEST_CLASS_IDX[forget_class]             # 1 000 test samples
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]                                                            # 9 000 test samples
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)

        tag = f'class{forget_class}_seed{seed}'
        if TEST_MODE: tag += '_testmode'
        ckpt_path = f'{CKPT_ROOT}/oracle_{tag}.pt'

        # ── Check for existing checkpoint (resumable) ────────────────────────────
        if os.path.exists(ckpt_path):
            print(f'[{tag}] exists — loading.')
            ck = torch.load(ckpt_path, map_location=device)
            all_results.append(ck['metrics'])
            print(f'  output R={ck["metrics"]["output_retain_acc"]:.2f}%  '
                  f'F={ck["metrics"]["output_forget_acc"]:.2f}%')
            continue

        # ── Train oracle from scratch on retain set only ─────────────────────────
        print(f'\n[{tag}] forget_class={forget_class}  '
              f'retain={len(retain_train_idx)}  forget={len(forget_train_idx)}')
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

        model = build_model()
        optimizer = optim.SGD(model.parameters(), lr=LR_INIT,
                              momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True)
        scheduler = make_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS, MIN_LR)

        # Validation loader: last 5000 samples of the retain set (no data leakage)
        val_subset = torch.utils.data.Subset(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            range(max(0, len(retain_train_idx) - 5000), len(retain_train_idx))
        )
        val_loader = torch.utils.data.DataLoader(
            val_subset, batch_size=256, shuffle=False, num_workers=2)

        best_val   = 0.0
        best_state = None
        patience_cnt = 0
        early_stopped_at = None
        log = []

        t_start = time.time()
        for epoch in range(1, EPOCHS + 1):
            model.train()
            for x, y in retain_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                F.cross_entropy(model(x), y).backward()
                optimizer.step()
            scheduler.step()

            val_acc = eval_acc(model, val_loader)
            log.append({'epoch': epoch, 'val_acc': val_acc})
            if val_acc > best_val:
                best_val   = val_acc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
            if epoch % 50 == 0:
                print(f'  epoch {epoch:3d}: val={val_acc:.2f}%  patience={patience_cnt}/{PATIENCE}')
            if patience_cnt >= PATIENCE:
                early_stopped_at = epoch
                print(f'  Early stop at epoch {epoch}')
                break

        wall_min = (time.time() - t_start) / 60.0
        model.load_state_dict(best_state)

        # ── Final evaluation — matches paper Table 1 exactly ─────────────────────
        # Output : test-set forget/retain accuracy
        # Probe  : linear head trained on ALL 50k training features → test set
        # NCC    : class means from ALL 50k training features → test set
        metrics = eval_three_metrics(
            model,
            test_retain_ldr,       # test-set retain  (output + probe eval + NCC eval)
            test_forget_ldr,       # test-set forget  (output + probe eval + NCC eval)
            train_retain_eval_ldr, # training-set retain (probe/NCC feature pool)
            train_forget_eval_ldr  # training-set forget (probe/NCC feature pool)
        )
        metrics.update({
            'model': 'oracle', 'forget_class': forget_class, 'seed': seed,
            'wall_clock_minutes': wall_min,
            'early_stopped_at': early_stopped_at,
            'n_forget_train': len(forget_train_idx),
            'n_retain_train': len(retain_train_idx),
            'protocol': 'whole_class_single',
        })

        torch.save({
            'model_state_dict': model.state_dict(),
            'config': {
                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                'forget_class': forget_class, 'seed': seed,
                'lr_init': LR_INIT, 'weight_decay': WEIGHT_DECAY,
                'momentum': MOMENTUM, 'warmup_epochs': WARMUP_EPOCHS,
                'epochs': EPOCHS, 'patience': PATIENCE, 'min_lr': MIN_LR,
                'protocol': 'whole_class_single',
                'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
            },
            'seed': seed,
            'metrics': metrics,
            'train_log': log,
        }, ckpt_path)
        print(f'  Saved {ckpt_path}')
        print(f'  output  R={metrics["output_retain_acc"]:.2f}%  F={metrics["output_forget_acc"]:.2f}%')
        print(f'  probe   R={metrics["probe_retain_acc"]:.2f}%  F={metrics["probe_forget_acc"]:.2f}%')
        print(f'  ncc     R={metrics["ncc_retain_acc"]:.2f}%  F={metrics["ncc_forget_acc"]:.2f}%')
        all_results.append(metrics)


# ── Aggregate: mean ± std over all forget classes (matches paper Table 1) ────────
df = pd.DataFrame(all_results)
csv_path = f'{CKPT_ROOT}/results_oracle_paper.csv'
df.to_csv(csv_path, index=False)
print(f'\nAll done. Per-run results saved: {csv_path}')

if df.empty:
    print('WARNING: no successful runs.')
else:
    metric_cols = [
        'output_retain_acc', 'output_forget_acc',
        'probe_retain_acc',  'probe_forget_acc',
        'ncc_retain_acc',    'ncc_forget_acc',
    ]
    summary = (
        df[metric_cols]
        .agg(['mean', 'std'])
        .round(2)
    )
    summary_path = f'{CKPT_ROOT}/results_oracle_paper_summary.csv'
    summary.to_csv(summary_path)
    print(f'Summary (mean±std over {len(FORGET_CLASSES)} forget classes) saved: {summary_path}')
    print('\n=== Mean across all forget classes ===')
    print(summary.loc['mean'].to_string())